In [1]:
import pandas as pd
import os

In [43]:
def rewrite_all_results(root, ban_list, fnum, fused):
    top = False
    if top == True:
        num = 12
    all_results = []
    counter = 0
    if fused:
        for disease in os.listdir(root):
            if disease.startswith('ICD') and disease.split('_')[-1].split('.')[0] not in ban_list:
                result_df = pd.read_csv(os.path.join(root,disease))
                if len(result_df) == fnum:
                    counter += 1
                    mean_df = result_df.groupby(['method','para'])[['top_recall_25','top_recall_300','top_recall_10%', 'top_precision_10%', 'max_precision_10%','top_recall_30%', 'top_precision_30%', 'max_precision_30%','pm_0.5%','pm_1%','pm_5%','pm_10%','pm_15%','pm_20%','pm_25%','pm_30%','auroc',"rank_ratio",'bedroc_1','bedroc_5','bedroc_10','bedroc_30']].mean().reset_index()
                    # Add disease information
                    mean_df['disease'] = disease.split('.')[0]
                    # Append to all_results list
                    all_results.append(mean_df)
                else:
                    print(disease, 'not enough feature')
            if top == True:
                if counter == num:
                    break
        # Concatenate all results into a single DataFrame
        final_result = pd.concat(all_results, ignore_index=True)
        if top == True:
            final_result.to_csv(os.path.join(root,f'all_disease_{num}.csv'),index=False)
        else:
            final_result.to_csv(os.path.join(root,'all_disease.csv'),index=False)
        return final_result
    else:
        for disease in os.listdir(root):
            if disease.startswith('ICD'):
                counter += 1
                result_df = pd.read_csv(os.path.join(root,disease))
                mean_df = result_df.groupby(['method'])[['top_recall_25','top_recall_300','top_recall_10%', 'top_precision_10%', 'max_precision_10%','top_recall_30%', 'top_precision_30%', 'max_precision_30%','pm_0.5%','pm_1%','pm_5%','pm_10%','pm_15%','pm_20%','pm_25%','pm_30%','auroc',"rank_ratio",'bedroc_1','bedroc_5','bedroc_10','bedroc_30']].mean().reset_index()
                # Add disease information
                mean_df['disease'] = disease.split('.')[0]
                # Append to all_results list
                all_results.append(mean_df)
            if top == True:
                if counter == num:
                    break
        # Concatenate all results into a single DataFrame
        final_result = pd.concat(all_results, ignore_index=True)
        if top == True:
            final_result.to_csv(os.path.join(root,f'all_disease_{num}.csv'),index=False)
        else:
            final_result.to_csv(os.path.join(root,'all_disease.csv'),index=False)
        return final_result

def create_summary(results,col_name, fused):
    # Create an empty list to store results
    summary_list = []
    if fused == False:
        # Grouping by 'method' and calculating mean and std for selected metrics
        for method, subdf in results.groupby(col_name):
            num_cols = subdf.select_dtypes(include='number').columns
            mean_values = subdf[num_cols].mean()
            std_values  = subdf[num_cols].std()
            summary_list.append({
                'method': method,
                'top_recall_25_mean': mean_values['top_recall_25'], 'top_recall_25_std': std_values['top_recall_25'],
                'top_recall_300_mean': mean_values['top_recall_300'], 'top_recall_300_std': std_values['top_recall_300'],
                'top_recall_10%_mean': mean_values['top_recall_10%'], 'top_recall_10%_std': std_values['top_recall_10%'],
                'top_precision_10_mean': mean_values['top_precision_10%'], 'top_precision_10_std': std_values['top_precision_10%'],
                'max_precision_10_mean': mean_values['max_precision_10%'], 'max_precision_10_std': std_values['max_precision_10%'],
                'top_recall_30_mean': mean_values['top_recall_30%'], 'top_recall_30_std': std_values['top_recall_30%'],
                'top_precision_30_mean': mean_values['top_precision_30%'], 'top_precision_10_std': std_values['top_precision_30%'],
                'max_precision_30_mean': mean_values['max_precision_30%'], 'max_precision_10_std': std_values['max_precision_30%'],
                'pm_0.5%': mean_values['pm_0.5%'], 'pm_0.5%_std': std_values['pm_0.5%'],
                'pm_1%': mean_values['pm_1%'], 'pm_1%_std': std_values['pm_1%'],
                'pm_5%': mean_values['pm_5%'], 'pm_5%_std': std_values['pm_5%'],
                'pm_10%': mean_values['pm_10%'], 'pm_10%_std': std_values['pm_10%'],
                'pm_15%': mean_values['pm_15%'], 'pm_15%_std': std_values['pm_15%'],            
                'pm_20%': mean_values['pm_20%'], 'pm_20%_std': std_values['pm_20%'],
                'pm_25%': mean_values['pm_25%'], 'pm_1%_std': std_values['pm_25%'],
                'pm_30%': mean_values['pm_30%'], 'pm_30%_std': std_values['pm_30%'],
                'auroc_mean': mean_values['auroc'], 'auroc_std': std_values['auroc'],
                'rank_ratio_mean': mean_values['rank_ratio'], 'rank_ratio_std': std_values['rank_ratio'],
                'bedroc_1_mean': mean_values['bedroc_1'], 'bedroc_1_std': std_values['bedroc_1'],
                'bedroc_5_mean': mean_values['bedroc_5'], 'bedroc_5_std': std_values['bedroc_5'],
                'bedroc_10_mean': mean_values['bedroc_10'], 'bedroc_10_std': std_values['bedroc_10'],
                'bedroc_30_mean': mean_values['bedroc_30'], 'bedroc_30_std': std_values['bedroc_30'],
                'weights_1_mean': mean_values['weights_1'], 'weights_1_std': std_values['weights_1'],
                'weights_2_mean': mean_values['weights_2'], 'weights_2_std': std_values['weights_2'],
                'weights_3_mean': mean_values['weights_3'], 'weights_3_std': std_values['weights_3']
            })

        # Convert the list of dictionaries into a DataFrame
        summary_df = pd.DataFrame(summary_list)
    else:
                # Grouping by 'method' and calculating mean and std for selected metrics
        for paras, subdf in results.groupby(col_name):
            num_cols = subdf.select_dtypes(include='number').columns
            mean_values = subdf[num_cols].mean()
            std_values  = subdf[num_cols].std()
            summary_list.append({
                'method': paras[0],
                'para': paras[1],
                'top_recall_25_mean': mean_values['top_recall_25'], 'top_recall_25_std': std_values['top_recall_25'],
                'top_recall_300_mean': mean_values['top_recall_300'], 'top_recall_300_std': std_values['top_recall_300'],
                'top_recall_10%_mean': mean_values['top_recall_10%'], 'top_recall_10%_std': std_values['top_recall_10%'],
                'top_precision_10_mean': mean_values['top_precision_10%'], 'top_precision_10_std': std_values['top_precision_10%'],
                'max_precision_10_mean': mean_values['max_precision_10%'], 'max_precision_10_std': std_values['max_precision_10%'],
                'top_recall_30_mean': mean_values['top_recall_30%'], 'top_recall_30_std': std_values['top_recall_30%'],
                'top_precision_30_mean': mean_values['top_precision_30%'], 'top_precision_10_std': std_values['top_precision_30%'],
                'max_precision_30_mean': mean_values['max_precision_30%'], 'max_precision_10_std': std_values['max_precision_30%'],
                'pm_0.5%': mean_values['pm_0.5%'], 'pm_0.5%_std': std_values['pm_0.5%'],
                'pm_1%': mean_values['pm_1%'], 'pm_1%_std': std_values['pm_1%'],
                'pm_5%': mean_values['pm_5%'], 'pm_5%_std': std_values['pm_5%'],
                'pm_10%': mean_values['pm_10%'], 'pm_10%_std': std_values['pm_10%'],
                'pm_15%': mean_values['pm_15%'], 'pm_15%_std': std_values['pm_15%'],            
                'pm_20%': mean_values['pm_20%'], 'pm_20%_std': std_values['pm_20%'],
                'pm_25%': mean_values['pm_25%'], 'pm_1%_std': std_values['pm_25%'],
                'pm_30%': mean_values['pm_30%'], 'pm_30%_std': std_values['pm_30%'],
                'auroc_mean': mean_values['auroc'], 'auroc_std': std_values['auroc'],
                'rank_ratio_mean': mean_values['rank_ratio'], 'rank_ratio_std': std_values['rank_ratio'],
                'bedroc_1_mean': mean_values['bedroc_1'], 'bedroc_1_std': std_values['bedroc_1'],
                'bedroc_5_mean': mean_values['bedroc_5'], 'bedroc_5_std': std_values['bedroc_5'],
                'bedroc_10_mean': mean_values['bedroc_10'], 'bedroc_10_std': std_values['bedroc_10'],
                'bedroc_30_mean': mean_values['bedroc_30'], 'bedroc_30_std': std_values['bedroc_30'],
                'weights_1_mean': mean_values['weights_1'], 'weights_1_std': std_values['weights_1'],
                'weights_2_mean': mean_values['weights_2'], 'weights_2_std': std_values['weights_2'],
                'weights_3_mean': mean_values['weights_3'], 'weights_3_std': std_values['weights_3']
            })

        # Convert the list of dictionaries into a DataFrame
        summary_df = pd.DataFrame(summary_list)
    return summary_df

In [3]:
from IPython.display import display, HTML

In [44]:
def show_table(root, ban_list, fnum, fused, input_weights):
    if fused:
        final_result = rewrite_all_results(root, ban_list, fnum, fused=True)
        if input_weights:
            weight_df = final_result.copy()
            
            weight_df[['para', 'weights_1', 'weights_2', 'weights_3']] = (
                weight_df['para'].str.split('-', expand=True)
            )

            # convert the weight columns to float
            weight_df[['weights_1', 'weights_2', 'weights_3']] = weight_df[['weights_1', 'weights_2', 'weights_3']].astype(float)

            all_sum = create_summary(weight_df, ['method', 'para'], fused=True)
            show_df = all_sum.sort_values(by='method', ascending=True) \
                .loc[:, all_sum.columns.str.contains(
                    'method|para|top_recall_300_mean|top_recall_10%_mean|auroc_mean|rank_ratio_mean|bedroc_1_mean|bedroc_5_mean|bedroc_10_mean|bedroc_30_mean|weights_1_mean|weights_2_mean|weights_3_mean', case=False)] \
                .round(3).rename(columns=lambda x: x.replace('_mean', ''))
            display(HTML(show_df.to_html(index=False).replace('<table', '<table style="font-size:13px; white-space:nowrap;"')))
            return weight_df, show_df
        else:
            all_sum = create_summary(final_result, ['method', 'para'], fused=True)
            show_df = all_sum.sort_values(by='method', ascending=True) \
                .loc[:, all_sum.columns.str.contains(
                    'method|para|top_recall_300_mean|top_recall_10%_mean|auroc_mean|bedroc_1_mean|bedroc_5_mean|bedroc_10_mean|bedroc_30_mean|rank_ratio', case=False)] \
                .round(3).rename(columns=lambda x: x.replace('_mean', ''))
            display(HTML(show_df.to_html(index=False).replace('<table', '<table style="font-size:13px; white-space:nowrap;"')))
            return final_result, show_df
    else:
        final_result = rewrite_all_results(root, fused=False)
        all_sum = create_summary(final_result, 'method', fused=False)
        if 'random_pos_negative_bagging' in all_sum['method'].unique():
            method_order = ['random_negativeauroc', 'random_negative_bagging', 'random_pos_negative_bagging']
            all_sum['method'] = pd.Categorical(all_sum['method'], categories=method_order, ordered=True)
        
        show_df = all_sum.sort_values(by='method', ascending=True) \
            .loc[:, all_sum.columns.str.contains(
                'method|auroc_mean|bedroc_1_mean|bedroc_5_mean|bedroc_10_mean|bedroc_30_mean|rank_ratio', case=False)] \
            .round(3).rename(columns=lambda x: x.replace('_mean', ''))
        
        display(HTML(show_df.to_html(index=False).replace('<table', '<table style="font-size:13px; white-space:nowrap;"')))
        return final_result, show_df

    

icd_dict = {
    'Certain infectious and parasitic diseases': ['A00','B99'],
    'Neoplasms': ['C00','D48'],
    'Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism': ['D50','D89'],
    'Endocrine, nutritional and metabolic diseases': ['E00','E90'],
    'Mental and behavioural disorders': ['F00','F99'],
    'Diseases of the nervous system': ['G00','G99'],
    'Diseases of the eye and adnexa': ['H00','H59'],
    'Diseases of the ear and mastoid process': ['H60','H95'],
    'Diseases of the circulatory system': ['I00','I99'],
    'Diseases of the respiratory system': ['J00','J99'],
    'Diseases of the digestive system': ['K00','K93'],
    'Diseases of the skin and subcutaneous tissue': ['L00','L99'],
    'Diseases of the musculoskeletal system and connective tissue': ['M00','M99'],
    'Diseases of the genitourinary system': ['N00','N99']
}

def find_disease_category(icd_code):
    icd_num = icd_code.split('_')[1]  # Extract ICD-10 code
    # print(icd_num)
    icd_letter = icd_num[0]  # Extract first letter (C, D, etc.)
    icd_number = int(icd_num[1:])  # Extract numeric part

    for category, (start, end) in icd_dict.items():
        start_letter, start_num = start[0], int(start[1:])
        end_letter, end_num = end[0], int(end[1:])

        if start_letter <= icd_letter <= end_letter:  # Ensure it's within the letter range
            if start_letter == icd_letter and start_num <= icd_number:
                return category
            if end_letter == icd_letter and icd_number <= end_num:
                return category
            if start_letter < icd_letter < end_letter:
                return category  # Covers ranges like C00-D48
        
    return 'Unknown Category'

def disease_catygory(results):
    mapped_results = {icd: find_disease_category(icd) for icd in results['disease']}
    results['category'] = results['disease'].map(mapped_results)

    collected_dfs = []
    disease_num = dict()
    for category in results['category'].unique().tolist():
        subdf = results[results['category'] == category].copy()
        sum_df = create_summary(subdf, 'method', fused=False)
        sum_df['category'] = category
        collected_dfs.append(sum_df)
        disease_num[category] = len(subdf) / 2

    final_df = pd.concat(collected_dfs, ignore_index=True)

    category_order = final_df.groupby('category', observed=True)['auroc_mean'].mean().sort_values(ascending=False).index.tolist()
    final_df['category'] = pd.Categorical(final_df['category'], categories=category_order, ordered=True)

    show_df = final_df.sort_values(by=['category', 'auroc_mean'], ascending=[True, False]) \
        .loc[:, final_df.columns.str.contains('method|para|top_recall_300_mean|top_recall_10%_mean|auroc_mean|rank_ratio_mean |bedroc_1_mean|bedroc_5_mean|bedroc_10_mean|bedroc_30_mean|weights_1_mean|weights_2_mean|weights_2_mean', case=False)] \
        .round(3) \
        .rename(columns=lambda x: x.replace('_mean', ''))

    # Highlight specific method
    def highlight_method(val):
        if isinstance(val, str) and val == 'random_pos_negative_bagging':
            return '<span style="color:red; font-weight:bold;">random_pos_negative_bagging</span>'
        return val

    styled_df = show_df.copy()
    if 'method' in styled_df.columns:
        styled_df['method'] = styled_df['method'].apply(highlight_method)

    # Assign background colors per category
    category_colors = {}
    base_colors = ['#ffdddd', "#dbf7db"]
    for i, cat in enumerate(category_order):
        category_colors[cat] = base_colors[i % len(base_colors)]

    def row_style(row):
        color = category_colors.get(row['category'], '#ffffff')
        return [f'background-color: {color}'] * len(row)


    # Display styled table
    from IPython.display import display, HTML
    display(HTML(styled_df.style.apply(row_style, axis=1).to_html(escape=False)))


# def disease_catygory_fused(results):
#     mapped_results = {icd: find_disease_category(icd) for icd in results['disease']}
#     results['category'] = results['disease'].map(mapped_results)

#     collected_dfs = []
#     disease_num = dict()
#     for category in results['category'].unique().tolist():
#         subdf = results[results['category'] == category].copy()
#         subdf = subdf.drop(columns='weights')
#         sum_df = create_summary(subdf, ['method', 'para'], fused=True)
#         sum_df['category'] = category
#         collected_dfs.append(sum_df)
#         disease_num[category] = len(subdf) / 2

#     final_df = pd.concat(collected_dfs, ignore_index=True)

#     category_order = final_df.groupby('category', observed=True)['auroc_mean'].mean().sort_values(ascending=False).index.tolist()
#     final_df['category'] = pd.Categorical(final_df['category'], categories=category_order, ordered=True)

#     show_df = final_df.sort_values(by=['category', 'auroc_mean'], ascending=[True, False]) \
#         .loc[:, final_df.columns.str.contains('method|para|auroc_mean|bedroc_1_mean|bedroc_5_mean|bedroc_10_mean|bedroc_30_mean|category', case=False)] \
#         .round(3) \
#         .rename(columns=lambda x: x.replace('_mean', ''))

#     # Define the custom order for 'feature'
#     feature_order = ['ppi_2019', 'bioconcept', 'esm2', 'uniport', 
#                     'linear_fused', 'geo_fused','weighted_linear_fused','weighted_geo_fused']

#     # Convert 'feature' to a categorical type with that order
#     show_df['para'] = pd.Categorical(show_df['para'], categories=feature_order, ordered=True)

#     # Now sort by 'disease' first, then 'feature' by the custom order
#     show_df = show_df.sort_values(by=['category', 'para'])
    
#     # Define alternating background colors per disease
#     base_colors = ['#ffdddd', '#dbf7db']
#     target_cols = ['auroc', 'bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30', 'weights']

#     # Build mapping for background colors per disease
#     diseases = show_df['disease'].unique()
#     disease_color_map = {disease: base_colors[i % len(base_colors)] for i, disease in enumerate(diseases)}

#     # Function to create full styling DataFrame
#     def combined_style(df):
#         styles = pd.DataFrame('', index=df.index, columns=df.columns)

#         # Add background color row-wise
#         for idx, row in df.iterrows():
#             bg_color = disease_color_map[row['disease']]
#             styles.loc[idx, :] = f'background-color: {bg_color};'

#         # Highlight max in each group & each target column
#         for disease, group in df.groupby('disease'):
#             for col in target_cols:
#                 max_val = group[col].max()
#                 max_indices = group[group[col] == max_val].index
#                 for idx in max_indices:
#                     styles.loc[idx, col] += ' color: red; font-weight: bold;'
        
#         return styles

#     # Apply combined style
#     styled_df = show_df.style.apply(combined_style, axis=None)
#     return styled_df

In [45]:
def prcess_and_save_xlsx(fused_2019,all_avg_df,out_path):
    full_fused_2019 = fused_2019.round(3)
    mapped_results = {icd: find_disease_category(icd) for icd in full_fused_2019['disease']}
    full_fused_2019['category'] = full_fused_2019['disease'].map(mapped_results)
    # full_fused_2019 = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/results/weighted_fused_2019.csv')
    full_fused_2019 = full_fused_2019[['method', 'para', 'top_recall_300','top_recall_10%','auroc','rank_ratio','bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30', 'weights_1', 'weights_2', 'weights_3','disease', 'category']]

    full_fused_2019 = full_fused_2019.round(3)

    later_fused = [item for item in full_fused_2019['para'].unique().tolist() if 'nor_' in item or 'later' in item]
    
    feature_order = ['uniport_ppi_2019', 'ppi_2019_dw_40','diffusion_2019','uniport_bio', 'uniport_esm', 'uniport_seq', 
                    'linear_fused', 'geo_fused','early_fusion'] + later_fused

    # Convert 'feature' to a categorical type with that order
    full_fused_2019['para'] = pd.Categorical(full_fused_2019['para'], categories=feature_order, ordered=True)

    # Now sort by 'disease' first, then 'feature' by the custom order
    full_fused_2019 = full_fused_2019.sort_values(by=['disease', 'para'])

    target_cols = ['top_recall_300','top_recall_10%','auroc', 'bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30', 'weights_1', 'weights_2', 'weights_3']

    # Define alternating background colors per disease
    base_colors = ['#ffdddd', '#dbf7db']
    diseases = full_fused_2019['disease'].unique()
    disease_color_map = {disease: base_colors[i % len(base_colors)] for i, disease in enumerate(diseases)}

    def combined_style(df):
        styles = pd.DataFrame('', index=df.index, columns=df.columns)

        # Add background color row-wise
        for idx, row in df.iterrows():
            bg_color = disease_color_map[row['disease']]
            styles.loc[idx, :] = f'background-color: {bg_color};'

        # Highlight max in each group & each target column
        for category, group in df.groupby('disease'):
            for col in target_cols:
                max_val = group[col].max()
                max_indices = group[group[col] == max_val].index
                for idx in max_indices:
                    styles.loc[idx, col] += ' color: red; font-weight: bold;'
        
        return styles


    # Build mapping for background colors per disease
    # full_fused_2019['para'] = full_fused_2019['para'].str.replace('ppi_2016', 'ppi_2017', regex=False)
    # Identify numeric columns
    numeric_cols = full_fused_2019.select_dtypes(include=['number']).columns

    # Apply styling and format ONLY numeric columns
    full_fused_2019_disease = (
        full_fused_2019.style
        .apply(combined_style, axis=None)
        .format({col: "{:.3f}" for col in numeric_cols})
    )

    # fused_2019['para'] = fused_2019['para'].str.replace('ppi_2016', 'ppi_2017', regex=False)
    results = fused_2019
    mapped_results = {icd: find_disease_category(icd) for icd in results['disease']}
    results['category'] = results['disease'].map(mapped_results)

    collected_dfs = []
    disease_num = dict()
    for category in results['category'].unique().tolist():
        subdf = results[results['category'] == category].copy()
        # subdf = subdf.drop(columns='weights')
        sum_df = create_summary(subdf, ['method', 'para'], fused=True)
        sum_df['category'] = category
        collected_dfs.append(sum_df)
        disease_num[category] = len(subdf) / 2

    final_df = pd.concat(collected_dfs, ignore_index=True)

    category_order = final_df.groupby('category', observed=True)['auroc_mean'].mean().sort_values(ascending=False).index.tolist()
    final_df['category'] = pd.Categorical(final_df['category'], categories=category_order, ordered=True)

    show_df = final_df.sort_values(by=['category', 'auroc_mean'], ascending=[True, False]) \
        .loc[:, final_df.columns.str.contains('method|para|top_recall_300_mean|top_recall_10%_mean|auroc_mean|bedroc_1_mean|bedroc_5_mean|bedroc_10_mean|bedroc_30_mean|weights_1_mean|weights_2_mean|weights_3_mean|category', case=False)] \
        .round(3) \
        .rename(columns=lambda x: x.replace('_mean', ''))

    # feature_order = ['ppi_2017', 'gene2vec', 'esm2', 'uniport', 
    #                  'linear_fused', 'geo_fused','weighted_linear_fused','weighted_geo_fused']

    # Convert 'feature' to a categorical type with that order
    show_df['para'] = pd.Categorical(show_df['para'], categories=feature_order, ordered=True)

    # Now sort by 'disease' first, then 'feature' by the custom order
    show_df = show_df.sort_values(by=['category', 'para'])

    # Define alternating background colors per disease
    base_colors = ['#ffdddd', '#dbf7db']


    # Build mapping for background colors per disease
    category = show_df['category'].unique()
    color_map = {disease: base_colors[i % len(base_colors)] for i, disease in enumerate(category)}

    target_cols = ['top_recall_300','top_recall_10%','auroc', 'bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30','weights_1', 'weights_2', 'weights_3']

    def combined_style2(df):
        styles = pd.DataFrame('', index=df.index, columns=df.columns)

        # Add background color row-wise
        for idx, row in df.iterrows():
            bg_color = color_map[row['category']]
            styles.loc[idx, :] = f'background-color: {bg_color};'

        # Highlight max in each group & each target column
        for category, group in df.groupby('category'):
            for col in target_cols:
                max_val = group[col].max()
                max_indices = group[group[col] == max_val].index
                for idx in max_indices:
                    styles.loc[idx, col] += ' color: red; font-weight: bold;'
        
        return styles


    numeric_cols = show_df.select_dtypes(include=['number']).columns

    # Apply styling and format ONLY numeric columns
    styled_df = (
        show_df.style
        .apply(combined_style2, axis=None)
        .format({col: "{:.3f}" for col in numeric_cols})
    )

    valid_feature_order = [f for f in feature_order if f in all_avg_df['para'].unique()]
    all_avg_df = all_avg_df.set_index('para').loc[valid_feature_order].reset_index()

    def highlight_max_font_red(s):
        is_max = s == s.max()
        return ['color: red; font-weight: bold' if v else '' for v in is_max]
    all_avg_df =all_avg_df[['para', 'top_recall_300', 'top_recall_10%','auroc', 'rank_ratio', 'bedroc_1',
        'bedroc_5', 'bedroc_10', 'bedroc_30','weights_1', 'weights_2', 'weights_3']].round(3)
    # Apply styling
    styled_all_avg_df = all_avg_df.style.apply(highlight_max_font_red, subset=target_cols)

    with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
        styled_all_avg_df.to_excel(writer, sheet_name='all (macro avg)', index=False)
        styled_df.to_excel(writer, sheet_name='category (macro avg)', index=False)
        full_fused_2019_disease.to_excel(writer, sheet_name='disease', index=False)

In [6]:
# import os
# import pandas as pd


# for file in os.listdir(root):
#     file_path = os.path.join(root, file)

#     # Skip non-CSV files
#     if not file.endswith(".csv"):
#         continue

#     single_df = pd.read_csv(file_path)

#     # Update 'para' column where '+' appears
#     single_df.loc[
#         single_df['para'].str.contains('+', regex=False, na=False),
#         'para'
#     ] = 'later_fused_alpha_'+ single_df['para'].str.split('+').str[0] + '-0'
#     # break
#     single_df.to_csv(file_path, index=False)


In [46]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_lf_bag'
out_path = root+'.xlsx'
temp_df = pd.read_csv(os.path.join(root,os.listdir(root)[2]))

fused_2019, all_avg_df = show_table(root,ban_list=[],fnum=len(temp_df),fused = True,input_weights = True)
prcess_and_save_xlsx(fused_2019,all_avg_df,out_path)

method,para,top_recall_300,top_recall_10%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,diffusion_2019,0.386,0.609,0.880,0.121,0.174,0.371,0.478,0.672,0.156,0.019,0.160
random_negative,uniport_esm,0.112,0.302,0.663,0.337,0.066,0.150,0.215,0.379,0.144,0.015,0.144
random_negative,uniport_bio,0.186,0.419,0.735,0.266,0.086,0.206,0.286,0.460,0.158,0.021,0.161
random_negative,ppi_2019_dw_40,0.315,0.585,0.825,0.175,0.140,0.304,0.403,0.588,0.170,0.016,0.174
random_negative,later_fused_ppi,0.346,0.673,0.861,0.145,0.178,0.356,0.464,0.653,0.162,0.020,0.168
random_negative,later_fused_bag_2_uniport_seq,0.162,0.322,0.679,0.323,0.079,0.180,0.242,0.398,0.140,0.015,0.139
random_negative,later_fused_bag_2_uniport_ppi_2019,0.304,0.558,0.810,0.191,0.145,0.305,0.405,0.592,0.164,0.017,0.169
random_negative,later_fused_bag_2_uniport_esm,0.144,0.306,0.666,0.337,0.075,0.162,0.224,0.384,0.140,0.016,0.141
random_negative,later_fused_bag_2_uniport_bio,0.200,0.429,0.735,0.266,0.093,0.215,0.292,0.463,0.158,0.022,0.161
random_negative,later_fused_bag_2_ppi_2019_dw_40,0.317,0.583,0.827,0.174,0.146,0.311,0.410,0.592,0.169,0.016,0.173


/tmp/ipykernel_673675/3189241166.py:112: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


## merge cv

In [79]:
from features_reindex import get_feature, read_data, read_data_timecut
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import re

time = 2019
if time == 2019:
    feature_list = ['uniport_ppi_2019','ppi_2019_dw_40','uniport_bio','uniport_seq','uniport_esm']
elif time == 2017:
    feature_list = ['uniport_ppi_2017','ppi_2017_dw_80','uniport_exp','uniport_seq','uniport_esm']

results_df = pd.read_excel(out_path, sheet_name='disease')
log_path = root.replace('results', 'src')+'.log'

code_root = '/itf-fi-ml/shared/users/ziyuzh/svm'

merged_df = None
for feature in feature_list:
    feature_df = get_feature(code_root, feature)
    # Rename columns starting with 'feature'
    feature_df.rename(columns={
        col: f"{feature}_{col}" if col.startswith('feature') else col
        for col in feature_df.columns
    }, inplace=True)

    feature_cols = [col for col in feature_df.columns if col.startswith('feature')]
    if feature_cols:
        scaler = MinMaxScaler()
        feature_df[feature_cols] = scaler.fit_transform(feature_df[feature_cols])

    # Merge iteratively to avoid keeping all DataFrames
    if merged_df is None:
        merged_df = feature_df
    else:
        merged_df = pd.merge(merged_df, feature_df, on='string_id', how='inner')
    del feature_df  # Free memory

all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/dga_time_uniport.csv')
all_df = all_df[all_df['string_id'].isin(merged_df['string_id'])]

selected_diseases = []
data = []

for disease_id in all_df['disease_id'].unique():
    sub_df = all_df[all_df['disease_id'] == disease_id]
    if len(sub_df) < 15:
        continue
    else:
        max_year = sub_df['first_pub_year'].max()
        min_year = sub_df['first_pub_year'].min()
        count_before = len(sub_df[sub_df['first_pub_year'] <= time])
        count_after = len(sub_df[sub_df['first_pub_year'] > time])
        
        if max_year > time and min_year <= time and len(sub_df[sub_df['first_pub_year'] < time]) >= 5:
            selected_diseases.append(disease_id)
            data.append([disease_id, count_before, count_after])

# Create a DataFrame with the results
split_df = pd.DataFrame(data, columns=['disease', f'train_sum', f'test_sum'])

# results_df = results_df.merge(split_df, on='disease',how='left')
results_df = results_df.merge(split_df, on='disease',how='left')

In [80]:
########### auc0.8 / all fused
# Read from your text file
with open(log_path, 'r') as f:
    lines = f.readlines()

data = []
current_disease = None
valid_features = set()
buffered_entries = []
lines_iter = iter(lines)

for line in lines_iter:
    line = line.strip()

    # Skip headers like the first list of features
    if re.match(r"^\[.*\]\s+\d+\s+\d+$", line):
        continue

    # New disease block
    if re.match(r'^ICD10_', line):
        current_disease = line.split()[0]
        valid_features = set()
        buffered_entries = []
        continue

    # Collect valid features
    if line.startswith("collect valid feature:"):
        valid_features = set(eval(line.split(":", 1)[1].strip()))
        for entry in buffered_entries:
            entry["valid feature"] = entry["feature"] in valid_features
            data.append(entry)
        buffered_entries = []
        continue

    # ✅ Check for early fusion BEFORE general kernel match
    if "early fusion" == line.lower():
        try:
            next_line = next(lines_iter).strip()
            # print(next_line)
        except StopIteration:
            continue

        if 'C_num' in next_line:
            info = next_line.split(' ')
            c_num = info[2]
            gamma = info[6]
            bedroc10 = info[7]
            aucroc = info[8]

            data.append({
                "disease": current_disease,
                "feature": "early_fusion",
                "cv para": {
                    "C_num": c_num,
                    "gamma": round(float(gamma.strip("'}'")), 3)
                },
                "cv bedroc10": bedroc10,
                "cv aucroc": aucroc,
                "valid feature": False
            })
        continue

    # Kernel features
    match = re.match(r"^(\S+)\s+(\{[^}]+\})\s+([\d.eE+-]+)\s+([\d.eE+-]+)$", line)
    if match:
        feature_name = match.group(1)
        try:
            cv_para = eval(match.group(2))
        except (SyntaxError, NameError):
            continue
        bedroc10 = round(float(match.group(3)), 3)
        aucroc = round(float(match.group(4)), 3)
        buffered_entries.append({
            "disease": current_disease,
            "feature": feature_name,
            "cv para": {
                "C_num": cv_para.get("C_num"),
                "gamma": round(float(cv_para.get("gamma")), 3)
            },
            "cv bedroc10": bedroc10,
            "cv aucroc": aucroc,
            "valid feature": False
        })
        continue

    # Fusion models
    if re.match(r"^(linear_fused|geo_fused|weighted_linear_fused|weighted_geo_fused)", line):
        parts = line.split()
        if len(parts) >= 4:
            feature_name = parts[0]
            try:
                cv_para = {"C": float(parts[1])}
                bedroc10 = round(float(parts[2]), 3)
                aucroc = round(float(parts[3]), 3)
            except ValueError:
                continue
            data.append({
                "disease": current_disease,
                "feature": feature_name,
                "cv para": cv_para,
                "cv bedroc10": bedroc10,
                "cv aucroc": aucroc,
                "valid feature": False
            })

# Convert to DataFrame
cv_df = pd.DataFrame(data)


In [81]:
results_df.rename(columns={'para': 'feature'}, inplace=True)
results_df = results_df.merge(cv_df, on=['disease','feature'],how='left')
results_df.rename(columns={'weights': 'pathway enrich intersection (pred & train)'}, inplace=True)

target_cols = ['top_recall_300', 'top_recall_10%','auroc','rank_ratio', 'bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30']

# Define alternating background colors per disease
base_colors = ['#ffdddd', '#dbf7db']
diseases = results_df['disease'].unique()
disease_color_map = {disease: base_colors[i % len(base_colors)] for i, disease in enumerate(diseases)}

def combined_style(df):
    styles = pd.DataFrame('', index=df.index, columns=df.columns)

    # Add background color row-wise
    for idx, row in df.iterrows():
        bg_color = disease_color_map[row['disease']]
        styles.loc[idx, :] = f'background-color: {bg_color};'

    # Highlight max in each group & each target column
    for category, group in df.groupby('disease'):
        for col in target_cols:
            max_val = group[col].max()
            max_indices = group[group[col] == max_val].index
            for idx in max_indices:
                styles.loc[idx, col] += ' color: red; font-weight: bold;'
    
    return styles

numeric_cols = results_df.select_dtypes(include=['number']).columns

# Apply styling and format ONLY numeric columns
full_fused_2019_disease = (
    results_df.style
    .apply(combined_style, axis=None)
    .format({col: "{:.3f}" for col in numeric_cols})
)

with pd.ExcelWriter(out_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    full_fused_2019_disease.to_excel(writer, sheet_name='disease', index=False)

## micro and weighted auc

In [82]:
all_disease_df = pd.read_excel(out_path, sheet_name="disease")

all_data = []
sum_test = 0
for d in all_disease_df['disease'].unique():
    subdf = all_disease_df[all_disease_df['disease']==d]

    # Select only relevant metric columns
    weighted_df = subdf[['method', 'disease','feature', 'top_recall_300', 'top_recall_10%','auroc', 'rank_ratio', 'bedroc_1', 'bedroc_5',
                        'bedroc_10', 'bedroc_30']].copy()

    # Get test sample count for this disease (assuming 'subdf' column stores that)
    test_sum = subdf['test_sum'].unique()
    assert len(test_sum) == 1, "Multiple or no test sample counts found"
    test_sum = test_sum[0]
    sum_test+= test_sum
    # Multiply float-type metric columns by test_sum
    for col in weighted_df.columns:
        if weighted_df[col].dtype == 'float':
            weighted_df[col] = weighted_df[col] * test_sum
    all_data.append(weighted_df)
combined_df = pd.concat(all_data, ignore_index=True)
micro_list = []
for f in combined_df['feature'].unique():
    subdf = combined_df[combined_df['feature']==f]
    float_sums = subdf.select_dtypes(include='float').sum()/sum_test
    float_sums_df = pd.DataFrame([float_sums])
    float_sums_df['feature'] = f
    micro_list.append(float_sums_df)
weighted_avg_df = pd.concat(micro_list, ignore_index=True)

weighted_cat = []
for c in all_disease_df['category'].unique():
    all_data = []
    sum_test = 0
    disease_list = all_disease_df[all_disease_df['category']==c]['disease'].unique()
    for d in disease_list:
        subdf = all_disease_df[all_disease_df['disease']==d]

        # Select only relevant metric columns
        weighted_df = subdf[['method', 'disease','feature', 'top_recall_300', 'top_recall_10%','auroc', 'rank_ratio', 'bedroc_1', 'bedroc_5',
                            'bedroc_10', 'bedroc_30']].copy()

        # Get test sample count for this disease (assuming 'subdf' column stores that)
        test_sum = subdf['test_sum'].unique()
        assert len(test_sum) == 1, "Multiple or no test sample counts found"
        test_sum = test_sum[0]
        sum_test+= test_sum
        # Multiply float-type metric columns by test_sum
        for col in weighted_df.columns:
            if weighted_df[col].dtype == 'float':
                weighted_df[col] = weighted_df[col] * test_sum
        all_data.append(weighted_df)
    combined_df = pd.concat(all_data, ignore_index=True)
    micro_list = []
    for f in combined_df['feature'].unique():
        subdf = combined_df[combined_df['feature']==f]
        float_sums = subdf.select_dtypes(include='float').sum()/sum_test
        float_sums_df = pd.DataFrame([float_sums])
        float_sums_df['feature'] = f
        micro_list.append(float_sums_df)
    micro_df = pd.concat(micro_list, ignore_index=True)
    micro_df['category'] = c
    weighted_cat.append(micro_df)
weighted_cat_df = pd.concat(weighted_cat, ignore_index=True)

combined_df = weighted_cat_df[['feature', 'top_recall_300', 'top_recall_10%','auroc', 'rank_ratio','bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30','category']]
# Define alternating background colors per disease
base_colors = ['#ffdddd', '#dbf7db']


# Build mapping for background colors per disease
category = combined_df['category'].unique()
color_map = {disease: base_colors[i % len(base_colors)] for i, disease in enumerate(category)}

target_cols = ['top_recall_300', 'top_recall_10%','auroc','bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30']

def combined_style2(df):
    styles = pd.DataFrame('', index=df.index, columns=df.columns)

    # Add background color row-wise
    for idx, row in df.iterrows():
        bg_color = color_map[row['category']]
        styles.loc[idx, :] = f'background-color: {bg_color};'

    # Highlight max in each group & each target column
    for category, group in df.groupby('category'):
        for col in target_cols:
            max_val = group[col].max()
            max_indices = group[group[col] == max_val].index
            for idx in max_indices:
                styles.loc[idx, col] += ' color: red; font-weight: bold;'
    
    return styles


numeric_cols = combined_df.select_dtypes(include=['number']).columns

# Apply styling and format ONLY numeric columns
styled_df = (
    combined_df.style
    .apply(combined_style2, axis=None)
    .format({col: "{:.3f}" for col in numeric_cols})
)

with pd.ExcelWriter(out_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    styled_df.to_excel(writer, sheet_name='category (weighted avg)', index=False)
    weighted_avg_df.to_excel(writer, sheet_name='all (weighted avg)', index=False)

In [ ]:
# all_disease_df
# all_disease_df = pd.read_excel(out_path, sheet_name="disease")

# all_data = []
# sum_test = 0
# for d in all_disease_df['disease'].unique():
#     subdf = all_disease_df[all_disease_df['disease']==d]

#     # Select only relevant metric columns
#     weighted_df = subdf[['method', 'disease','feature','rank_ratio']].copy()

#     # Get test sample count for this disease (assuming 'subdf' column stores that)
#     test_sum = subdf['test_sum'].unique()
#     assert len(test_sum) == 1, "Multiple or no test sample counts found"
#     test_sum = test_sum[0]
#     sum_test+= test_sum
#     # Multiply float-type metric columns by test_sum
#     for col in weighted_df.columns:
#         if weighted_df[col].dtype == 'float':
#             weighted_df[col] = weighted_df[col] * test_sum
#     all_data.append(weighted_df)

    
# combined_df = pd.concat(all_data, ignore_index=True)
# micro_list = []
# for f in combined_df['feature'].unique():
#     subdf = combined_df[combined_df['feature']==f]
#     float_sums = subdf.select_dtypes(include='float').sum()/sum_test
#     float_sums_df = pd.DataFrame([float_sums])
#     float_sums_df['feature'] = f
#     micro_list.append(float_sums_df)
# weighted_avg_df = pd.concat(micro_list, ignore_index=True)